In [16]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
import sys
!{sys.executable} -m pip install seaborn



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program F

In [17]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [18]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [19]:
import pandas as pd
import numpy as np
#import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [20]:
import pandas as pd
import awswrangler as wr

# Parámetros
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Ruta S3 del parquet
s3_path = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA/data_pn_total_expandido_new.parquet"

# Cargar parquet desde S3
df_inference = wr.s3.read_parquet(path=s3_path, boto3_session=session)

print(f"✓ Parquet cargado desde S3")
print(f"  Shape: {df_inference.shape}")
print(f"  Columnas: {df_inference.columns.tolist()[:10]}...")  # primeras 10
display(df_inference.head())

✓ Parquet cargado desde S3
  Shape: (1685224, 65)
  Columnas: ['target', 'key_value', 'cod_cli', 'codmes_lag1', 'cod_mes', 'mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'max_trx_abonos_3m']...


,target,key_value,cod_cli,codmes_lag1,cod_mes,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,share_cp_ingresos,rat_cntros_x_cnttrxegr_3m,alertas_por_antiguedad,rat_ing_tot_x_factura_6m,rat_pastot_x_ingtot_6m,ratio_egresos_exterior,rat_ing_ext_x_ing_tot_12m,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente
0,0,FBE607E8BAF83AA079A7A05BE7352B9C2CE44C4EEDCC61...,0016672468,202501,202502,449.38,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,SIN_INFO,SIN_INFO
1,0,DCBB6FDE45B91530D08956E350412F7FF2A2E4738E5D92...,0021169720,202504,202505,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,SIN_INFO,SIN_INFO
2,0,F649EC21B2C76591F6373EF9CE2B4D87CF8137DA9F4EDA...,0016634616,202506,202507,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,SIN_INFO,SIN_INFO
3,0,43F3D0996F1169674BBA3AB7BE06C827E390F48CDEA25A...,0015487500,202501,202502,6.75,0.00,40330.00,1.33,2218.00,...,0.00,0.00,0.00,3.08,0.00,0.00,0.00,0.00,SIN_INFO,SIN_INFO
4,0,9173CB33F4392E61064DF8A8A2642AE5C2B27345363791...,0021543611,202506,202507,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,SIN_INFO,SIN_INFO


In [21]:
df_inference.shape

(1685224, 65)

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL   = "target"
PERIODO_COL  = "cod_mes"
PERIODOS     = [202501, 202502, 202503, 202504, 202505, 202506, 202507]
OUTPUT_DIR   = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\bivariados_graficos"
N_BINS       = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Variables seleccionadas ──────────────────────────────────────────────────
VARIABLES_SELECCIONADAS = [
    'mto_pas_soles',
 'imp_trx_abonosefect_6m',
 'imp_trx_cargosefe_6m',
 'avg_trx_cargostot_3m',
 'cnt_trx_cargostot_3m',
 'cnt_trx_abonospromtot_3m',
 'rat_trx_abonosefectot_1m',
 'rat_trx_abonosefectot_3m',
 'rat_trx_abonosefectot_9m',
 'rat_mntcrgsefetot_1m',
 'num_antiguedad',
 'cnt_meses_sinegresos_12m',
 'cod_ubigeo_cd',
 'cod_sectorista_id',
 'flg_vrcn_abonos_5m_1m',
 'flg_vrcn_efe_cargos_5m_1m',
 'mto_fact_declarado_sunat',
 'avg_cp_men_ing_12m',
 'avg_cpmenegr_12m',
 'max_mto_cpegrmen_12m',
 'cnt_trx_sinenv_alext_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'cnt_noticias',
 'flg_alerta_12m',
 'cnt_alerta_hist',
 'cnt_ros_hist',
 'ratio_abonos_1m_vs_6m',
 'ratio_cargos_1m_vs_6m',
 'share_cp_egresos',
 'share_cp_ingresos',
 'rat_cntros_x_cnttrxegr_3m',
 'ingresos_vs_facturacion',
 'rat_pastot_x_ingtot_6m',
 'ratio_egresos_exterior',
 'rat_ing_ext_x_ing_tot_12m',
]

# ── Filtrar periodos ─────────────────────────────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
print(f"✓ Registros filtrados: {df.shape[0]:,}  |  Target rate: {df[TARGET_COL].mean():.4f}")

# ── Validar columnas existentes ──────────────────────────────────────────────
variables     = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas en el DataFrame: {no_existentes}")
print(f"✓ Variables a graficar: {len(variables)}")

# ── Función de análisis bivariado ────────────────────────────────────────────
def plot_bivariado(df, col, target, n_bins, output_dir):
    df_tmp = df[[col, target]].copy()
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

    try:
        df_tmp["bin"] = pd.qcut(df_tmp[col], q=n_bins, duplicates="drop")
    except Exception:
        df_tmp["bin"] = pd.cut(df_tmp[col], bins=n_bins)

    resumen = (
        df_tmp.groupby("bin", observed=True)[target]
        .agg(count="count", target_rate="mean")
        .reset_index()
    )
    resumen["bin_str"] = resumen["bin"].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.bar(resumen["bin_str"], resumen["count"], color="#4C72B0", alpha=0.7, label="Registros")
    ax1.set_xlabel(col, fontsize=11)
    ax1.set_ylabel("Registros", fontsize=11, color="#4C72B0")
    ax1.tick_params(axis="x", rotation=45, labelsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    ax2 = ax1.twinx()
    ax2.plot(resumen["bin_str"], resumen["target_rate"], color="#DD4949",
             marker="o", linewidth=2, label="Target rate")
    ax2.set_ylabel("Target rate", fontsize=11, color="#DD4949")
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    plt.title(f"Bivariado: {col}  |  periodos {PERIODOS[0]}–{PERIODOS[-1]}", fontsize=13)
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"biv_{col}.png")
    plt.savefig(filepath, dpi=120, bbox_inches="tight")
    plt.close()
    return filepath

# ── Loop principal ───────────────────────────────────────────────────────────
guardados = []
errores   = []

for i, col in enumerate(variables, 1):
    try:
        fp = plot_bivariado(df, col, TARGET_COL, N_BINS, OUTPUT_DIR)
        guardados.append(fp)
        print(f"  [{i}/{len(variables)}] {col} ✓")
    except Exception as e:
        errores.append((col, str(e)))

print(f"\n✅ Gráficos guardados: {len(guardados)}")
print(f"⚠  Errores:           {len(errores)}")
if errores:
    for col, err in errores:
        print(f"   → {col}: {err}")
print(f"\n📁 Carpeta: {OUTPUT_DIR}")

✓ Registros filtrados: 175,400  |  Target rate: 0.0050
⚠ Columnas no encontradas en el DataFrame: ['ratio_abonos_1m_vs_6m', 'ingresos_vs_facturacion']
✓ Variables a graficar: 34
  [1/34] mto_pas_soles ✓
  [2/34] imp_trx_abonosefect_6m ✓
  [3/34] imp_trx_cargosefe_6m ✓
  [2/34] imp_trx_abonosefect_6m ✓
  [3/34] imp_trx_cargosefe_6m ✓
  [4/34] avg_trx_cargostot_3m ✓
  [5/34] cnt_trx_cargostot_3m ✓
  [4/34] avg_trx_cargostot_3m ✓
  [5/34] cnt_trx_cargostot_3m ✓
  [6/34] cnt_trx_abonospromtot_3m ✓
  [7/34] rat_trx_abonosefectot_1m ✓
  [6/34] cnt_trx_abonospromtot_3m ✓
  [7/34] rat_trx_abonosefectot_1m ✓
  [8/34] rat_trx_abonosefectot_3m ✓
  [9/34] rat_trx_abonosefectot_9m ✓
  [8/34] rat_trx_abonosefectot_3m ✓
  [9/34] rat_trx_abonosefectot_9m ✓
  [10/34] rat_mntcrgsefetot_1m ✓
  [11/34] num_antiguedad ✓
  [10/34] rat_mntcrgsefetot_1m ✓
  [11/34] num_antiguedad ✓
  [12/34] cnt_meses_sinegresos_12m ✓
  [12/34] cnt_meses_sinegresos_12m ✓
  [13/34] cod_ubigeo_cd ✓
  [13/34] cod_ubigeo_cd ✓
  [

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
#import seaborn as sns
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL  = "target"
PERIODO_COL = "cod_mes"
PERIODOS    = [202501, 202502, 202503, 202504, 202505, 202506, 202507]
OUTPUT_DIR  = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\eda_graficos"
N_BINS      = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

VARIABLES_SELECCIONADAS = [
    'mto_pas_soles',
 'imp_trx_abonosefect_6m',
 'imp_trx_cargosefe_6m',
 'avg_trx_cargostot_3m',
 'cnt_trx_cargostot_3m',
 'cnt_trx_abonospromtot_3m',
 'rat_trx_abonosefectot_1m',
 'rat_trx_abonosefectot_3m',
 'rat_trx_abonosefectot_9m',
 'rat_mntcrgsefetot_1m',
 'num_antiguedad',
 'cnt_meses_sinegresos_12m',
 'cod_ubigeo_cd',
 'cod_sectorista_id',
 'flg_vrcn_abonos_5m_1m',
 'flg_vrcn_efe_cargos_5m_1m',
 'mto_fact_declarado_sunat',
 'avg_cp_men_ing_12m',
 'avg_cpmenegr_12m',
 'max_mto_cpegrmen_12m',
 'cnt_trx_sinenv_alext_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'cnt_noticias',
 'flg_alerta_12m',
 'cnt_alerta_hist',
 'cnt_ros_hist',
 'ratio_abonos_1m_vs_6m',
 'ratio_cargos_1m_vs_6m',
 'share_cp_egresos',
 'share_cp_ingresos',
 'rat_cntros_x_cnttrxegr_3m',
 'ingresos_vs_facturacion',
 'rat_pastot_x_ingtot_6m',
 'ratio_egresos_exterior',
 'rat_ing_ext_x_ing_tot_12m',
]

# ── Filtrar periodos y columnas existentes ───────────────────────────────────
df = df_inference[df_inference[PERIODO_COL].astype(int).isin(PERIODOS)].copy()
variables = [c for c in VARIABLES_SELECCIONADAS if c in df.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas: {no_existentes}")

# Convertir a numérico
for col in variables:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df_eda = df[variables + [TARGET_COL]].copy()

print(f"✓ Shape EDA: {df_eda.shape}")
print(f"✓ Target rate: {df_eda[TARGET_COL].mean():.4f}")


# ════════════════════════════════════════════════════════════════════════════
# 1. RESUMEN ESTADÍSTICO
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("1. RESUMEN ESTADÍSTICO")
print("="*60)

stats = df_eda[variables].describe().T
stats["missing"]     = df_eda[variables].isnull().sum()
stats["missing_pct"] = (df_eda[variables].isnull().mean() * 100).round(2)
stats["zeros_pct"]   = ((df_eda[variables] == 0).mean() * 100).round(2)
stats["skewness"]    = df_eda[variables].skew().round(3)
stats["kurtosis"]    = df_eda[variables].kurt().round(3)

display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max",
               "missing_pct", "zeros_pct", "skewness", "kurtosis"]])

stats.to_csv(os.path.join(OUTPUT_DIR, "01_resumen_estadistico.csv"))
print(f"✓ Guardado: 01_resumen_estadistico.csv")


# ════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("2. MISSING VALUES")
print("="*60)

missing = df_eda[variables].isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

if missing.empty:
    print("✓ No hay missing values")
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing) * 0.35)))
    missing.plot(kind="barh", ax=ax, color="#E07B54")
    ax.set_xlabel("% Missing", fontsize=11)
    ax.set_title("Missing Values por Variable (%)", fontsize=13)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    for i, v in enumerate(missing):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_missing_values.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"✓ Guardado: 02_missing_values.png")
    display(missing.to_frame("missing_pct"))


# ════════════════════════════════════════════════════════════════════════════
# 3. DISTRIBUCIONES (histogramas)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("3. DISTRIBUCIONES")
print("="*60)

n_cols  = 4
n_rows  = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    data = df_eda[col].dropna()
    axes[i].hist(data, bins=40, color="#4C72B0", alpha=0.75, edgecolor="white")
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.1f}")
    ))

# Ocultar ejes vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuciones de Variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_distribuciones.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 03_distribuciones.png")


# ════════════════════════════════════════════════════════════════════════════
# 4. CORRELACIÓN CON EL TARGET
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("4. CORRELACIÓN CON EL TARGET")
print("="*60)

corr_target = (
    df_eda[variables + [TARGET_COL]]
    .corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_target) * 0.35)))
colors = ["#DD4949" if v > 0 else "#4C72B0" for v in corr_target]
corr_target.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson", fontsize=11)
ax.set_title(f"Correlación de Variables con '{TARGET_COL}'", fontsize=13)
for i, v in enumerate(corr_target):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_correlacion_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 04_correlacion_target.png")

display(corr_target.to_frame("corr_con_target").style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
corr_target.to_frame("corr_con_target").to_csv(os.path.join(OUTPUT_DIR, "04_correlacion_target.csv"))


# Reemplaza SOLO la sección 5 - MATRIZ DE CORRELACIÓN
# ════════════════════════════════════════════════════════════════════════════
# 5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES (sin seaborn)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES")
print("="*60)

corr_matrix = df_eda[variables].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_plot = corr_matrix.copy()
corr_plot[mask] = np.nan  # ocultar triángulo superior

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(corr_plot, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.6)

# Etiquetas
ax.set_xticks(range(len(variables)))
ax.set_yticks(range(len(variables)))
ax.set_xticklabels(variables, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(variables, fontsize=8)

# Anotaciones numéricas
for i in range(len(variables)):
    for j in range(len(variables)):
        val = corr_plot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=5.5, color="black" if abs(val) < 0.7 else "white")

ax.set_title("Matriz de Correlación entre Variables", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_matriz_correlacion.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 05_matriz_correlacion.png")

# Pares con alta correlación (> 0.7)
corr_upper = corr_matrix.where(mask == False).stack().reset_index()
corr_upper.columns = ["var1", "var2", "correlacion"]
corr_upper = corr_upper[corr_upper["var1"] != corr_upper["var2"]]
alta_corr  = corr_upper[corr_upper["correlacion"].abs() > 0.7].sort_values("correlacion", key=abs, ascending=False)

print(f"\n⚠ Pares con correlación > 0.7:")
display(alta_corr)
alta_corr.to_csv(os.path.join(OUTPUT_DIR, "05_alta_correlacion.csv"), index=False)


# ════════════════════════════════════════════════════════════════════════════
# 6. BOXPLOT TARGET=0 vs TARGET=1
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("6. DISTRIBUCIÓN POR TARGET (0 vs 1)")
print("="*60)

n_cols = 4
n_rows = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    g0 = df_eda[df_eda[TARGET_COL] == 0][col].dropna()
    g1 = df_eda[df_eda[TARGET_COL] == 1][col].dropna()
    axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
                    patch_artist=True,
                    boxprops=dict(facecolor="#4C72B0", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2))
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución por Target (0 vs 1)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_boxplot_por_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 06_boxplot_por_target.png")


# ════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : {len(variables):<25}║
║  Registros            : {df_eda.shape[0]:<25,}║
║  Target rate          : {df_eda[TARGET_COL].mean():<25.4f}║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")

⚠ Columnas no encontradas: ['ratio_abonos_1m_vs_6m', 'ingresos_vs_facturacion']
✓ Shape EDA: (175400, 35)
✓ Target rate: 0.0050

1. RESUMEN ESTADÍSTICO


,count,mean,std,min,25%,50%,75%,max,missing_pct,zeros_pct,skewness,kurtosis
mto_pas_soles,175400.00,72627.52,4201550.69,-17.08,0.00,157.95,4222.42,1204376054.77,0.00,29.99,180.62,43162.91
imp_trx_abonosefect_6m,175400.00,30250.23,498347.07,0.00,0.00,0.00,608.50,141888138.96,0.00,70.35,153.39,38856.95
imp_trx_cargosefe_6m,175400.00,30542.33,278016.57,0.00,0.00,0.00,4211.85,28018443.80,0.00,64.11,37.81,2290.92
avg_trx_cargostot_3m,175400.00,0.71,2.26,0.00,0.00,0.00,0.33,111.67,0.00,72.32,7.38,105.88
cnt_trx_cargostot_3m,175400.00,47.92,560.47,0.00,1.00,10.00,39.00,83211.00,0.00,23.47,105.52,12976.21
cnt_trx_abonospromtot_3m,175400.00,15.45,1514.81,0.00,0.00,1.00,4.00,369508.00,0.00,32.39,235.88,56490.83
rat_trx_abonosefectot_1m,175400.00,0.05,0.19,0.00,0.00,0.00,0.00,1.00,0.00,88.13,4.00,15.42
rat_trx_abonosefectot_3m,175400.00,0.08,0.22,0.00,0.00,0.00,0.00,1.00,0.00,78.84,3.22,9.60
rat_trx_abonosefectot_9m,175400.00,0.11,0.24,0.00,0.00,0.00,0.07,1.00,0.00,64.05,2.66,6.21
rat_mntcrgsefetot_1m,175400.00,0.10,0.27,0.00,0.00,0.00,0.00,1.00,0.00,82.24,2.63,5.36


✓ Guardado: 01_resumen_estadistico.csv

2. MISSING VALUES
✓ Guardado: 02_missing_values.png


,missing_pct
cod_sectorista_id,89.26
mto_fact_declarado_sunat,75.18



3. DISTRIBUCIONES
✓ Guardado: 03_distribuciones.png

4. CORRELACIÓN CON EL TARGET
✓ Guardado: 03_distribuciones.png

4. CORRELACIÓN CON EL TARGET
✓ Guardado: 04_correlacion_target.png
✓ Guardado: 04_correlacion_target.png


...


5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES
✓ Guardado: 05_matriz_correlacion.png

⚠ Pares con correlación > 0.7:
✓ Guardado: 05_matriz_correlacion.png

⚠ Pares con correlación > 0.7:


,var1,var2,correlacion
189,max_mto_cpegrmen_12m,avg_cpmenegr_12m,0.79
35,rat_trx_abonosefectot_9m,rat_trx_abonosefectot_3m,0.73
27,rat_trx_abonosefectot_3m,rat_trx_abonosefectot_1m,0.70



6. DISTRIBUCIÓN POR TARGET (0 vs 1)


C:\Users\b46637\AppData\Local\Temp\ipykernel_23044\1024941327.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_23044\1024941327.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_23044\1024941327.py:249: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[i].boxplot([g0, g1], labels=["target=0", "target=1"],
C:\Users\b46637\AppData\Local\Temp\ipykernel_23044\1024941327.py:249: MatplotlibDeprecationWarning: 

✓ Guardado: 06_boxplot_por_target.png

╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : 34                       ║
║  Registros            : 175,400                  ║
║  Target rate          : 0.0050                   ║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\eda_graficos



In [26]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# SELECCIÓN DE VARIABLES PARA EL MODELO
# Criterios:
#   1. Missing % < MAX_MISSING_PCT  (hasta 99% de nulos permitido)
#   2. Sin multicolinealidad: |corr entre variables| <= CORR_THRESHOLD
#      → se elimina la de MENOR correlación con el target
# ═══════════════════════════════════════════════════════════════════════════

MAX_MISSING_PCT = 99.0   # elimina solo variables con 100% nulos
CORR_THRESHOLD  = 0.70   # correlación máxima permitida entre variables

print("=" * 65)
print("SELECCIÓN DE VARIABLES PARA EL MODELO")
print("=" * 65)
print(f"  Umbral missing   : < {MAX_MISSING_PCT}%  (se eliminan solo las que tienen 100% nulos)")
print(f"  Umbral corr vars : <= {CORR_THRESHOLD}")
print(f"  Variables iniciales: {len(variables)}\n")

# ── 1. Filtro por missing ────────────────────────────────────────────────────
missing_pct = df_eda[variables].isnull().mean() * 100
vars_ok_missing   = missing_pct[missing_pct < MAX_MISSING_PCT].index.tolist()
vars_drop_missing = missing_pct[missing_pct >= MAX_MISSING_PCT].index.tolist()

print(f"── Filtro 1: MISSING >= {MAX_MISSING_PCT}%  →  eliminadas {len(vars_drop_missing)}")
for v in vars_drop_missing:
    print(f"   ✗ {v:<45}  ({missing_pct[v]:.1f}% nulos)")
print(f"   Quedan: {len(vars_ok_missing)} variables\n")

# ── 2. Correlación con el target ─────────────────────────────────────────────
corr_con_target = (
    df_eda[vars_ok_missing + [TARGET_COL]]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL, errors="ignore")
    .abs()
    .fillna(0)
)

# ── 3. Filtro multicolinealidad (greedy, conserva la más predictiva) ──────────
corr_matrix = df_eda[vars_ok_missing].corr(numeric_only=True).abs()
vars_num    = corr_matrix.columns.tolist()

eliminadas_corr = []
seleccionadas   = sorted(vars_num, key=lambda v: corr_con_target.get(v, 0), reverse=True)

i = 0
while i < len(seleccionadas):
    v1 = seleccionadas[i]
    j  = i + 1
    while j < len(seleccionadas):
        v2 = seleccionadas[j]
        if corr_matrix.loc[v1, v2] > CORR_THRESHOLD:
            eliminadas_corr.append((v2, v1, round(corr_matrix.loc[v1, v2], 3)))
            seleccionadas.pop(j)
        else:
            j += 1
    i += 1

vars_no_num = [v for v in vars_ok_missing if v not in vars_num]

print(f"── Filtro 2: CORRELACIÓN > {CORR_THRESHOLD}  →  eliminadas {len(eliminadas_corr)}")
for v_elim, v_keep, corr_val in eliminadas_corr:
    print(f"   ✗ {v_elim:<45}  (corr={corr_val:.3f} con '{v_keep}')")
print(f"   Quedan: {len(seleccionadas)} variables numéricas\n")

if vars_no_num:
    print(f"── Variables categóricas/string (sin filtro corr): {vars_no_num}\n")

# ── Lista final ───────────────────────────────────────────────────────────────
VARIABLES_MODELO = seleccionadas + vars_no_num

# ── Tabla resumen con selector interactivo ────────────────────────────────────
df_selector = pd.DataFrame({
    "variable"    : VARIABLES_MODELO,
    "missing_pct" : [round(missing_pct.get(v, 0), 2) for v in VARIABLES_MODELO],
    "corr_target" : [round(corr_con_target.get(v, float("nan")), 6) for v in VARIABLES_MODELO],
    "tipo"        : ["numérica" if v in vars_num else "categórica" for v in VARIABLES_MODELO],
    "incluir"     : [True] * len(VARIABLES_MODELO),   # puedes editar aquí para excluir manualmente
}).sort_values("corr_target", ascending=False).reset_index(drop=True)

print("=" * 65)
print(f"✅ VARIABLES SELECCIONADAS: {len(VARIABLES_MODELO)}")
print("=" * 65)

display(
    df_selector.style
    .background_gradient(subset=["corr_target"], cmap="Greens")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .format({"missing_pct": "{:.1f}%", "corr_target": "{:.6f}"})
    .set_properties(**{"font-size": "11px"})
)

# ── Guardar tabla de selección ────────────────────────────────────────────────
OUTPUT_SEL = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\variables_modelo.csv"
df_selector.to_csv(OUTPUT_SEL, index=False)
print(f"\n✓ Tabla guardada en: {OUTPUT_SEL}")

# ── Aplicar columna 'incluir' para obtener lista final editable ───────────────
VARIABLES_MODELO_FINAL = df_selector.loc[df_selector["incluir"] == True, "variable"].tolist()

print(f"\n📋 Lista final ({len(VARIABLES_MODELO_FINAL)} variables):")
print("VARIABLES_MODELO_FINAL = [")
for v in VARIABLES_MODELO_FINAL:
    print(f"    '{v}',")
print("]")

SELECCIÓN DE VARIABLES PARA EL MODELO
  Umbral missing   : < 99.0%  (se eliminan solo las que tienen 100% nulos)
  Umbral corr vars : <= 0.7
  Variables iniciales: 34

── Filtro 1: MISSING >= 99.0%  →  eliminadas 0
   Quedan: 34 variables

── Filtro 2: CORRELACIÓN > 0.7  →  eliminadas 3
   ✗ rat_trx_abonosefectot_1m                       (corr=0.704 con 'rat_trx_abonosefectot_3m')
   ✗ rat_trx_abonosefectot_9m                       (corr=0.733 con 'rat_trx_abonosefectot_3m')
   ✗ max_mto_cpegrmen_12m                           (corr=0.790 con 'avg_cpmenegr_12m')
   Quedan: 31 variables numéricas

✅ VARIABLES SELECCIONADAS: 31
── Filtro 2: CORRELACIÓN > 0.7  →  eliminadas 3
   ✗ rat_trx_abonosefectot_1m                       (corr=0.704 con 'rat_trx_abonosefectot_3m')
   ✗ rat_trx_abonosefectot_9m                       (corr=0.733 con 'rat_trx_abonosefectot_3m')
   ✗ max_mto_cpegrmen_12m                           (corr=0.790 con 'avg_cpmenegr_12m')
   Quedan: 31 variables numéricas

✅ VA

,variable,missing_pct,corr_target,tipo,incluir
0,imp_trx_cargosefe_6m,0.0%,0.195419,numérica,True
1,imp_trx_abonosefect_6m,0.0%,0.138392,numérica,True
2,rat_ing_ext_x_ing_tot_12m,0.0%,0.124642,numérica,True
3,flg_alerta_12m,0.0%,0.121967,numérica,True
4,ratio_cargos_1m_vs_6m,0.0%,0.114005,numérica,True
5,avg_trx_cargostot_3m,0.0%,0.096628,numérica,True
6,rat_trx_abonosefectot_3m,0.0%,0.093952,numérica,True
7,cod_sectorista_id,89.3%,0.083740,numérica,True
8,mto_del_ext_12m,0.0%,0.073878,numérica,True
9,cnt_ros_hist,0.0%,0.068271,numérica,True



✓ Tabla guardada en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\variables_modelo.csv

📋 Lista final (31 variables):
VARIABLES_MODELO_FINAL = [
    'imp_trx_cargosefe_6m',
    'imp_trx_abonosefect_6m',
    'rat_ing_ext_x_ing_tot_12m',
    'flg_alerta_12m',
    'ratio_cargos_1m_vs_6m',
    'avg_trx_cargostot_3m',
    'rat_trx_abonosefectot_3m',
    'cod_sectorista_id',
    'mto_del_ext_12m',
    'cnt_ros_hist',
    'avg_cpmenegr_12m',
    'rat_mntcrgsefetot_1m',
    'mto_fact_declarado_sunat',
    'cnt_noticias',
    'avg_cp_men_ing_12m',
    'cnt_trx_cargostot_3m',
    'cnt_alerta_hist',
    'flg_vrcn_abonos_5m_1m',
    'cnt_trx_sinenv_alext_12m',
    'mto_al_ext_12m',
    'rat_cntros_x_cnttrxegr_3m',
    'num_antiguedad',
    'cod_ubigeo_cd',
    'mto_pas_soles',
    'cnt_meses_sinegresos_12m',
    'cnt_trx_abonospromtot_3m',
    'flg_vrcn_efe_cargos_5m_1m',
    'share_cp_ingresos',
    'share_cp_egresos',
    'rat_pastot_x_ingtot_6m',
    'ratio_eg

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
import os

# ═══════════════════════════════════════════════════════════════════════════
# SELECTOR POR IMPORTANCIA ACUMULADA (LightGBM)
# Conserva las variables que acumulan el UMBRAL_IMPORTANCIA de la importancia
# total del modelo.
# ═══════════════════════════════════════════════════════════════════════════

UMBRAL_IMPORTANCIA = 0.99   # 99% de importancia acumulada
OUTPUT_DIR_SEL     = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_train"

# ── Preparar datos de entrenamiento ─────────────────────────────────────────
# Usamos VARIABLES_MODELO_FINAL (salida de la celda anterior)
# Solo columnas numéricas (LightGBM acepta NaN pero no strings sin encode)
vars_lgb = [v for v in VARIABLES_MODELO_FINAL
            if v in df_eda.select_dtypes(include=[np.number]).columns]

print(f"Variables numéricas para LightGBM : {len(vars_lgb)}")
print(f"Variables categóricas excluidas   : "
      f"{[v for v in VARIABLES_MODELO_FINAL if v not in vars_lgb]}\n")

X = df_eda[vars_lgb].copy()
y = df_eda[TARGET_COL].copy()

# Eliminar filas donde el target es NaN
mask_valid = y.notna()
X, y = X[mask_valid], y[mask_valid]

print(f"✓ Shape para entrenamiento: {X.shape}  |  Target rate: {y.mean():.5f}")

# ── Entrenar LightGBM rápido (solo para importancia) ────────────────────────
scale_pos = (y == 0).sum() / max((y == 1).sum(), 1)

params = {
    "objective"        : "binary",
    "metric"           : "auc",
    "n_estimators"     : 300,
    "learning_rate"    : 0.05,
    "num_leaves"       : 63,
    "min_child_samples": 50,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "scale_pos_weight" : scale_pos,
    "n_jobs"           : -1,
    "random_state"     : 42,
    "verbose"          : -1,
}

print("\n⏳ Entrenando LightGBM para calcular importancia de variables...")
model = lgb.LGBMClassifier(**params)
model.fit(X, y)
print("✓ Entrenamiento completado")

# ── Importancia acumulada ───────────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=vars_lgb)
importances = importances.sort_values(ascending=False)
importancias_norm = importances / importances.sum()
importancia_acum  = importancias_norm.cumsum()

# Variables que acumulan hasta UMBRAL_IMPORTANCIA
vars_99 = importancia_acum[importancia_acum <= UMBRAL_IMPORTANCIA].index.tolist()

# Asegurarse de incluir la variable que cruza el umbral (si quedaron justo por debajo)
if len(vars_99) < len(vars_lgb):
    siguiente = importancia_acum.index[len(vars_99)]
    vars_99.append(siguiente)

print(f"\n✅ Variables que acumulan {UMBRAL_IMPORTANCIA*100:.0f}% de importancia: {len(vars_99)} / {len(vars_lgb)}")

# ── Tabla resultado ─────────────────────────────────────────────────────────
df_importancia = pd.DataFrame({
    "variable"           : importancias_norm.index,
    "importancia"        : importancias_norm.values,
    "importancia_acum"   : importancia_acum.values,
    "missing_pct"        : [round(missing_pct.get(v, 0), 2) for v in importancias_norm.index],
    "seleccionada"       : [v in vars_99 for v in importancias_norm.index],
}).reset_index(drop=True)

display(
    df_importancia.style
    .background_gradient(subset=["importancia"], cmap="Greens")
    .background_gradient(subset=["importancia_acum"], cmap="Blues")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .apply(lambda col: ["background-color: #d4edda" if v else "" for v in df_importancia["seleccionada"]], axis=0)
    .format({
        "importancia"      : "{:.4%}",
        "importancia_acum" : "{:.2%}",
        "missing_pct"      : "{:.1f}%",
    })
    .set_properties(**{"font-size": "11px"})
)

# ── Gráfico importancia acumulada ───────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(max(14, len(vars_lgb) * 0.25), 6))

bars = ax1.bar(range(len(importancias_norm)), importancias_norm.values,
               color=["#2ecc71" if v in vars_99 else "#bdc3c7" for v in importancias_norm.index],
               alpha=0.85, label="Importancia individual")
ax1.set_xticks(range(len(importancias_norm)))
ax1.set_xticklabels(importancias_norm.index, rotation=90, fontsize=7)
ax1.set_ylabel("Importancia relativa", fontsize=11)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2 = ax1.twinx()
ax2.plot(range(len(importancia_acum)), importancia_acum.values,
         color="#e74c3c", linewidth=2, marker=".", markersize=4, label="Importancia acumulada")
ax2.axhline(UMBRAL_IMPORTANCIA, color="#e74c3c", linestyle="--", linewidth=1,
            label=f"Umbral {UMBRAL_IMPORTANCIA*100:.0f}%")
ax2.set_ylabel("Importancia acumulada", fontsize=11, color="#e74c3c")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right", fontsize=9)

plt.title(f"Importancia de Variables (LightGBM)  |  {len(vars_99)} vars explican {UMBRAL_IMPORTANCIA*100:.0f}% de importancia",
          fontsize=13)
plt.tight_layout()
fp_imp = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.png")
plt.savefig(fp_imp, dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Gráfico guardado: {fp_imp}")

# ── Guardar tabla ───────────────────────────────────────────────────────────
fp_csv = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.csv")
df_importancia.to_csv(fp_csv, index=False)
print(f"✓ Tabla guardada : {fp_csv}")

# ── Lista final para el modelo ──────────────────────────────────────────────
VARIABLES_FINALES_MODELO = vars_99

print(f"\n{'='*65}")
print(f"📋 VARIABLES_FINALES_MODELO ({len(VARIABLES_FINALES_MODELO)} variables)")
print(f"{'='*65}")
print("VARIABLES_FINALES_MODELO = [")
for v in VARIABLES_FINALES_MODELO:
    imp = importancias_norm.get(v, 0)
    print(f"    '{v}',  # importancia={imp:.4%}")
print("]")


Variables numéricas para LightGBM : 31
Variables categóricas excluidas   : []

✓ Shape para entrenamiento: (175400, 31)  |  Target rate: 0.00500

⏳ Entrenando LightGBM para calcular importancia de variables...
✓ Entrenamiento completado

✅ Variables que acumulan 99% de importancia: 29 / 31
✓ Entrenamiento completado

✅ Variables que acumulan 99% de importancia: 29 / 31


,variable,importancia,importancia_acum,missing_pct,seleccionada
0,cod_ubigeo_cd,9.8011%,9.80%,0.0%,True
1,cnt_trx_cargostot_3m,8.1667%,17.97%,0.0%,True
2,mto_pas_soles,8.0699%,26.04%,0.0%,True
3,rat_pastot_x_ingtot_6m,7.9839%,34.02%,0.0%,True
4,cnt_trx_abonospromtot_3m,7.0054%,41.03%,0.0%,True
5,imp_trx_abonosefect_6m,5.8495%,46.88%,0.0%,True
6,imp_trx_cargosefe_6m,4.9086%,51.78%,0.0%,True
7,rat_trx_abonosefectot_3m,4.2957%,56.08%,0.0%,True
8,ratio_cargos_1m_vs_6m,4.2204%,60.30%,0.0%,True
9,cnt_meses_sinegresos_12m,4.2151%,64.52%,0.0%,True


✓ Gráfico guardado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_train\07_importancia_variables.png
✓ Tabla guardada : c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_train\07_importancia_variables.csv

📋 VARIABLES_FINALES_MODELO (29 variables)
VARIABLES_FINALES_MODELO = [
    'cod_ubigeo_cd',  # importancia=9.8011%
    'cnt_trx_cargostot_3m',  # importancia=8.1667%
    'mto_pas_soles',  # importancia=8.0699%
    'rat_pastot_x_ingtot_6m',  # importancia=7.9839%
    'cnt_trx_abonospromtot_3m',  # importancia=7.0054%
    'imp_trx_abonosefect_6m',  # importancia=5.8495%
    'imp_trx_cargosefe_6m',  # importancia=4.9086%
    'rat_trx_abonosefectot_3m',  # importancia=4.2957%
    'ratio_cargos_1m_vs_6m',  # importancia=4.2204%
    'cnt_meses_sinegresos_12m',  # importancia=4.2151%
    'num_antiguedad',  # importancia=3.9570%
    'mto_fact_declarado_sunat',  # importancia=3.6720%
    'cnt_alerta_hist',  # importancia=3.5484%
    'rat_mnt

In [28]:
=================================================================
📋 VARIABLES_FINALES_MODELO (32 variables)
=================================================================
VARIABLES_FINALES_MODELO = [
    'cod_ubigeo_cd',  # importancia=9.8011%
    'cnt_trx_cargostot_3m',  # importancia=8.1667%
    'mto_pas_soles',  # importancia=8.0699%
    'rat_pastot_x_ingtot_6m',  # importancia=7.9839%
    'cnt_trx_abonospromtot_3m',  # importancia=7.0054%
    'imp_trx_abonosefect_6m',  # importancia=5.8495%
    'imp_trx_cargosefe_6m',  # importancia=4.9086%
    'rat_trx_abonosefectot_3m',  # importancia=4.2957%
    'ratio_cargos_1m_vs_6m',  # importancia=4.2204%
    'cnt_meses_sinegresos_12m',  # importancia=4.2151%
    'num_antiguedad',  # importancia=3.9570%
    'mto_fact_declarado_sunat',  # importancia=3.6720%
    'cnt_alerta_hist',  # importancia=3.5484%
    'rat_mntcrgsefetot_1m',  # importancia=3.0269%
    'avg_trx_cargostot_3m',  # importancia=2.8978%
    'cod_sectorista_id',  # importancia=2.5753%
    'mto_del_ext_12m',  # importancia=2.0753%
    'rat_cntros_x_cnttrxegr_3m',  # importancia=2.0430%
    'rat_ing_ext_x_ing_tot_12m',  # importancia=1.6505%
    'avg_cpmenegr_12m',  # importancia=1.5376%
    'share_cp_egresos',  # importancia=1.5054%
    'avg_cp_men_ing_12m',  # importancia=0.9677%
    'cnt_ros_hist',  # importancia=0.9086%
    'flg_vrcn_abonos_5m_1m',  # importancia=0.9086%
    'flg_alerta_12m',  # importancia=0.8602%
    'mto_al_ext_12m',  # importancia=0.7151%
    'ratio_egresos_exterior',  # importancia=0.6882%
    'cnt_noticias',  # importancia=0.6720%
    'cnt_trx_sinenv_alext_12m',  # importancia=0.5699%
]

SyntaxError: invalid character '📋' (U+1F4CB) (3414130805.py, line 2)

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# MAPEO DE VARIABLES: Nombres Usuario → Nombres Dataset
# Identifica qué variables del modelo necesitan ser renombradas
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd

# Mapeo completo: VARIABLE_USUARIO -> VARIABLE_DATASET
MAPEO_VARIABLES = {
    'mes_base': 'cod_mes',
    'codunico': 'cod_cli',
    'tipo_documento': 'cod_tip_doc',
    'num_documento': 'key_value',
    'tipo_alerta_n2': 'tip_alerta',
    'ciiu_v4': 'cod_ciiu_v4',
    'ingresos_vs_facturacion': 'rat_ing_tot_x_factura_6m',
    'q_alerta_hist': 'cnt_alerta_hist',
    'trx_monto_abonos_6m_efectivo': 'imp_trx_abonosefect_6m',
    'pasivo_soles': 'mto_pas_soles',
    'sectorista_id': 'cod_sectorista_id',
    'pasivo_vs_ingresos': 'rat_pastot_x_ingtot_6m',
    'edad_constitucion': 'num_edad_constitucion',
    'trx_q_abonos_ratio_3m_efectivo_total': 'rat_trx_abonosefectot_3m',
    'antiguedad': 'num_antiguedad',
    'trx_q_cargos_3m_total': 'cnt_trx_cargostot_3m',
    'ros_por_trx_3m': 'rat_cntros_x_cnttrxegr_3m',
    'q_meses_egresos_0': 'cnt_meses_sinegresos_12m',
    'ratio_abonos_1m_vs_6m': 'rat_abonos_1m_vs_6m',
    'ubigeo_cd': 'cod_ubigeo_cd',
    'trx_monto_cargos_6m_efectivo': 'imp_trx_cargosefe_6m',
    'trx_q_abonos_ratio_9m_efectivo_total': 'rat_trx_abonosefectot_9m',
    'ratio_ingresos_exterior': 'rat_ing_ext_x_ing_tot_12m',
    'alertas_por_antiguedad': 'rat_cnt_alert_x_num_antig',
    'flag_alerta_12m': 'flg_alerta_12m',
    'trx_q_abonos_promedio_3m_total': 'cnt_trx_abonospromtot_3m',
    'provincia': 'desc_provincia',
    'facturacion': 'mto_fact_declarado_sunat',
    'q_ros_hist': 'cnt_ros_hist',
    'ratio_cargos_1m_vs_6m': 'rat_cargos_1m_vs_6m',
    'trx_monto_cargos_promedio_3m_total': 'avg_trx_cargostot_3m',
    'flag_ros_12m': 'flg_ros_12m',
    'monto_al_exterior_12m': 'mto_al_ext_12m',
    'fecha_ejecucion': 'ts_carga',
    'p_codmes': 'p_codmes',
}

# Variables del modelo final (32 variables)
VARIABLES_FINALES_MODELO = [
    'cod_ubigeo_cd',
    'cnt_trx_cargostot_3m',
    'pasivo_vs_ingresos',
    'mto_pas_soles',
    'ratio_abonos_1m_vs_6m',
    'cnt_trx_abonospromtot_3m',
    'imp_trx_abonosefect_6m',
    'imp_trx_cargosefe_6m',
    'cnt_meses_sinegresos_12m',
    'rat_trx_abonosefectot_3m',
    'ratio_cargos_1m_vs_6m',
    'num_antiguedad',
    'cnt_alerta_hist',
    'avg_trx_cargostot_3m',
    'mto_fact_declarado_sunat',
    'rat_mntcrgsefetot_1m',
    'ros_por_trx_3m',
    'cod_sectorista_id',
    'mto_del_ext_12m',
    'ingresos_vs_facturacion',
    'share_cp_egresos',
    'ratio_egresos_exterior',
    'ratio_ingresos_exterior',
    'max_mto_cpegrmen_12m',
    'avg_cpmenegr_12m',
    'cnt_ros_hist',
    'flg_alerta_12m',
    'avg_cp_men_ing_12m',
    'cnt_noticias',
    'share_cp_ingresos',
    'cnt_trx_sinenv_alext_12m',
    'mto_al_ext_12m',
]

# Crear mapa inverso (DATASET -> USUARIO)
MAPEO_INVERSO = {v: k for k, v in MAPEO_VARIABLES.items()}

print("\n" + "="*80)
print("ANÁLISIS: VARIABLES QUE NECESITAN SER RENOMBRADAS")
print("="*80)

# Identificar variables que necesitan renombrado
variables_a_renombrar = []
variables_sin_cambio = []

for var_dataset in VARIABLES_FINALES_MODELO:
    if var_dataset in MAPEO_INVERSO:
        var_usuario = MAPEO_INVERSO[var_dataset]
        variables_a_renombrar.append({
            'Variable Dataset (Actual)': var_dataset,
            'Variable Usuario (Original)': var_usuario,
            'Acción': 'RENOMBRAR'
        })
    else:
        variables_sin_cambio.append(var_dataset)

# Crear DataFrame
df_renombres = pd.DataFrame(variables_a_renombrar)

print(f"\n✅ VARIABLES QUE NECESITAN RENOMBRADO: {len(variables_a_renombrar)}\n")

if len(variables_a_renombrar) > 0:
    display(df_renombres.style.background_gradient(subset=['Variable Dataset (Actual)'], cmap='Greens'))
    
    print(f"\n📋 LISTA PARA COPIAR/PEGAR:")
    print("\nCódigo de mapeo:")
    print("```python")
    print("MAPEO_RENOMBRES = {")
    for _, row in df_renombres.iterrows():
        print(f"    '{row['Variable Dataset (Actual)']:<30}': '{row['Variable Usuario (Original)']:<35}',")
    print("}")
    print("```")
else:
    print("   ✓ No hay variables para renombrar")

print(f"\n\n❌ VARIABLES SIN CAMBIO: {len(variables_sin_cambio)}")
print(f"   (Ya están en el nombre del Dataset o no tienen mapeo)\n")
if variables_sin_cambio:
    for var in variables_sin_cambio:
        print(f"   • {var}")

# Resumen estadístico
print(f"\n{'='*80}")
print("📊 RESUMEN:")
print(f"{'='*80}")
print(f"""
Total de variables en modelo           : {len(VARIABLES_FINALES_MODELO)}
Variables que necesitan renombrado     : {len(variables_a_renombrar)} ({100*len(variables_a_renombrar)/len(VARIABLES_FINALES_MODELO):.1f}%)
Variables que quedan igual             : {len(variables_sin_cambio)} ({100*len(variables_sin_cambio)/len(VARIABLES_FINALES_MODELO):.1f}%)

🎯 ACCIÓN RECOMENDADA:
   1. Aplicar el mapeo de renombres ANTES de usar el modelo en inferencia
   2. Validar que el dataset de entrada tenga TODOS los nombres en formato "Variable Dataset"
   3. Después del renombrado, usar los nombres del modelo consistentemente
""")

# Guardar tabla
OUTPUT_CSV = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_train\mapeo_variables_renombres.csv"
df_renombres.to_csv(OUTPUT_CSV, index=False)
print(f"\n✓ Tabla guardada en: {OUTPUT_CSV}")



ANÁLISIS: VARIABLES QUE NECESITAN SER RENOMBRADAS

✅ VARIABLES QUE NECESITAN RENOMBRADO: 16



ValueError: could not convert string to float: 'cod_ubigeo_cd'


📋 LISTA PARA COPIAR/PEGAR:

Código de mapeo:
```python
MAPEO_RENOMBRES = {
    'cod_ubigeo_cd                 ': 'ubigeo_cd                          ',
    'cnt_trx_cargostot_3m          ': 'trx_q_cargos_3m_total              ',
    'mto_pas_soles                 ': 'pasivo_soles                       ',
    'cnt_trx_abonospromtot_3m      ': 'trx_q_abonos_promedio_3m_total     ',
    'imp_trx_abonosefect_6m        ': 'trx_monto_abonos_6m_efectivo       ',
    'imp_trx_cargosefe_6m          ': 'trx_monto_cargos_6m_efectivo       ',
    'cnt_meses_sinegresos_12m      ': 'q_meses_egresos_0                  ',
    'rat_trx_abonosefectot_3m      ': 'trx_q_abonos_ratio_3m_efectivo_total',
    'num_antiguedad                ': 'antiguedad                         ',
    'cnt_alerta_hist               ': 'q_alerta_hist                      ',
    'avg_trx_cargostot_3m          ': 'trx_monto_cargos_promedio_3m_total ',
    'mto_fact_declarado_sunat      ': 'facturacion                        ',

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DICCIONARIO DE MAPEO: VARIABLE USUARIO (VIEJO) → VARIABLE DATASET (NUEVO)
# Para usar en inferencia: df.rename(columns=MAPEO_COLUMNAS, inplace=True)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd

# Mapeo COMPLETO: VIEJO (Usuario) → NUEVO (Dataset)
MAPEO_COLUMNAS = {
    'mes_base': 'cod_mes',
    'codunico': 'cod_cli',
    'tipo_documento': 'cod_tip_doc',
    'num_documento': 'key_value',
    'tipo_alerta_n2': 'tip_alerta',
    'ciiu_v4': 'cod_ciiu_v4',
    'ingresos_vs_facturacion': 'rat_ing_tot_x_factura_6m',
    'q_alerta_hist': 'cnt_alerta_hist',
    'trx_monto_abonos_6m_efectivo': 'imp_trx_abonosefect_6m',
    'pasivo_soles': 'mto_pas_soles',
    'sectorista_id': 'cod_sectorista_id',
    'pasivo_vs_ingresos': 'rat_pastot_x_ingtot_6m',
    'edad_constitucion': 'num_edad_constitucion',
    'trx_q_abonos_ratio_3m_efectivo_total': 'rat_trx_abonosefectot_3m',
    'antiguedad': 'num_antiguedad',
    'trx_q_cargos_3m_total': 'cnt_trx_cargostot_3m',
    'ros_por_trx_3m': 'rat_cntros_x_cnttrxegr_3m',
    'q_meses_egresos_0': 'cnt_meses_sinegresos_12m',
    'ratio_abonos_1m_vs_6m': 'rat_abonos_1m_vs_6m',
    'ubigeo_cd': 'cod_ubigeo_cd',
    'trx_monto_cargos_6m_efectivo': 'imp_trx_cargosefe_6m',
    'trx_q_abonos_ratio_9m_efectivo_total': 'rat_trx_abonosefectot_9m',
    'ratio_ingresos_exterior': 'rat_ing_ext_x_ing_tot_12m',
    'alertas_por_antiguedad': 'rat_cnt_alert_x_num_antig',
    'flag_alerta_12m': 'flg_alerta_12m',
    'trx_q_abonos_promedio_3m_total': 'cnt_trx_abonospromtot_3m',
    'provincia': 'desc_provincia',
    'facturacion': 'mto_fact_declarado_sunat',
    'q_ros_hist': 'cnt_ros_hist',
    'ratio_cargos_1m_vs_6m': 'rat_cargos_1m_vs_6m',
    'trx_monto_cargos_promedio_3m_total': 'avg_trx_cargostot_3m',
    'flag_ros_12m': 'flg_ros_12m',
    'monto_al_exterior_12m': 'mto_al_ext_12m',
    'fecha_ejecucion': 'ts_carga',
    'p_codmes': 'p_codmes',
}

print("\n" + "="*80)
print("📋 DICCIONARIO DE MAPEO: VIEJO → NUEVO")
print("="*80)

print(f"\n✅ Total de renombres: {len(MAPEO_COLUMNAS)}\n")

print("Copia este código para usar en inferencia:\n")
print("```python")
print("# Paso 1: Renombrar columnas del dataset de entrada")
print("df_entrada.rename(columns=MAPEO_COLUMNAS, inplace=True)\n")
print("# Paso 2: Seleccionar solo las variables del modelo")
print("df_entrada = df_entrada[VARIABLES_FINALES_MODELO]\n")
print("# Paso 3: Hacer inferencia")
print("predicciones = modelo.predict(df_entrada)")
print("```\n")

# Mostrar tabla de referencia
df_mapeo = pd.DataFrame({
    'VIEJO (Usuario)': list(MAPEO_COLUMNAS.keys()),
    'NUEVO (Dataset)': list(MAPEO_COLUMNAS.values()),
})

print("Tabla de referencia:\n")
display(df_mapeo)

# Guardar como CSV
OUTPUT_MAPEO = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_train\MAPEO_COLUMNAS_VIEJO_NUEVO.csv"
df_mapeo.to_csv(OUTPUT_MAPEO, index=False)
print(f"\n✓ Tabla guardada en: {OUTPUT_MAPEO}")

# Imprimir como diccionario Python
print("\n" + "="*80)
print("DICCIONARIO PYTHON (copiar/pegar):")
print("="*80 + "\n")
print("MAPEO_COLUMNAS = {")
for viejo, nuevo in MAPEO_COLUMNAS.items():
    print(f"    '{viejo}': '{nuevo}',")
print("}")

# Variables del modelo final que NECESITAN renombrado
VARIABLES_A_RENOMBRAR_EN_MODELO = {
    'ubigeo_cd': 'cod_ubigeo_cd',
    'trx_q_cargos_3m_total': 'cnt_trx_cargostot_3m',
    'pasivo_soles': 'mto_pas_soles',
    'trx_q_abonos_promedio_3m_total': 'cnt_trx_abonospromtot_3m',
    'trx_monto_abonos_6m_efectivo': 'imp_trx_abonosefect_6m',
    'trx_monto_cargos_6m_efectivo': 'imp_trx_cargosefe_6m',
    'q_meses_egresos_0': 'cnt_meses_sinegresos_12m',
    'trx_q_abonos_ratio_3m_efectivo_total': 'rat_trx_abonosefectot_3m',
    'antiguedad': 'num_antiguedad',
    'q_alerta_hist': 'cnt_alerta_hist',
    'trx_monto_cargos_promedio_3m_total': 'avg_trx_cargostot_3m',
    'facturacion': 'mto_fact_declarado_sunat',
    'sectorista_id': 'cod_sectorista_id',
    'q_ros_hist': 'cnt_ros_hist',
    'flag_alerta_12m': 'flg_alerta_12m',
    'monto_al_exterior_12m': 'mto_al_ext_12m',
}

print("\n\n" + "="*80)
print("⚡ RESUMEN PARA INFERENCIA:")
print("="*80)

print(f"""
Variables del modelo final        : 32
Variables que necesitan renombrado: 16 (50%)
Variables que quedan igual        : 16 (50%)

🔧 PASOS PARA USAR EL MODELO EN INFERENCIA:

1️⃣  RENOMBRAR COLUMNAS:
    df_datos.rename(columns=MAPEO_COLUMNAS, inplace=True)

2️⃣  SELECCIONAR SOLO VARIABLES DEL MODELO:
    df_datos = df_datos[VARIABLES_FINALES_MODELO]

3️⃣  VALIDAR QUE NO HAY MISSING CRÍTICOS:
    assert not df_datos.isnull().all().any(), 'Hay columnas con 100% nulos'

4️⃣  HACER INFERENCIA:
    predicciones = modelo.predict(df_datos)
    probabilidades = modelo.predict_proba(df_datos)
""")

print("="*80)
